In [ ]:
%pip install -q qualtran cirq


In [ ]:
import os, sys, json, shutil, numpy as np, zipfile
from pathlib import Path

COMP = Path('/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation')
WORK = Path('/kaggle/working')
sys.path.insert(0, str(COMP / 'kickmix'))
sys.path.insert(0, str(COMP / 'verifiers'))
sys.path.insert(0, '/kaggle/input/datasets/evanpieser/qcompile-verifiable')
sys.path.insert(0, '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit')
sys.path.insert(0, '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit/verifier')

from starter.task3_circuits.build_circuits import build_submission
from dfthcsos.verify import verify as CV

bp_members = Path('/kaggle/input/datasets/busyaprime/33-qcc-v8-members')
sub_dir = WORK / 'submission'
sub_dir.mkdir(exist_ok=True)
(sub_dir / 'dfthcsos').mkdir(exist_ok=True)

shutil.copy2(bp_members / 'invsqrt.kmb', sub_dir / 'invsqrt.kmb')
shutil.copy2(bp_members / 'dqi.kmb', sub_dir / 'dqi.kmb')
for p in bp_members.glob('dfthcsos__*.kmb'):
    shutil.copy2(p, sub_dir / 'dfthcsos' / p.name.replace('dfthcsos__', ''))

opt_fact = Path('/kaggle/input/datasets/evanpieser/optimized-fact2/optimized.npz')
shutil.copy2(opt_fact, sub_dir / 'dfthcsos' / 'submission.npz')

print('Setup complete')


In [ ]:
result = build_submission(
    'FeMoCo76o113e', sub_dir / 'dfthcsos' / 'submission.npz',
    tree='log_depth', brot=8, batch_size=5, rounding='nearest',
    compiler='qrom',
    output_directory=sub_dir / 'dfthcsos', overwrite=True,
)
print('Build result score:', result['score'])
print('Passes:', result['metrics'] if 'metrics' in result else 'N/A')

r = CV(sub_dir / 'dfthcsos')
print('Full verify:', r['status'], r['score'])

zip_path = WORK / 'submission_optimized.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for p in sorted(sub_dir.rglob('*')):
        if p.is_file():
            z.write(p, str(p.relative_to(sub_dir)))
print('Wrote', zip_path, zip_path.stat().st_size, 'bytes')
